[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/09_formats_complexity.ipynb)

# Formats complexity — comparing modes of convening

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-16

**Abstract**: Public-dialogue studies often ask whether *how* people are convened changes
*what* they raise — workshop versus survey, chatbot versus in-person. Multilogue
does not hard-code “format”; it supplies the Address machinery with a group
column and a consumer metric $\ell$. This notebook teaches that pattern: Access
on a public format pair (CIMT dialogue vs survey when available), then a full
offline Address demo on format-tagged synthetic units with FakeClient, including
soft contrasts, document EMD, and a multiverse over analytic decisions.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

An `ai-dialogue-formats`-style study keeps PDF parsers, prompts, and the precise
definition of $\ell$ in the consumer package. Multilogue owns extract → cluster →
weight → soft lenses → document distance → multiverse. The group column here is
`format` $\in\{\mathrm{dialogue},\mathrm{survey}\}$. The headline $\ell$ is a
document-weighted format gap on the lens with largest absolute difference —
recomputed under each analytic configuration without new LLM calls.


In [ ]:
# Optional transport dependency (soft lenses + document EMD).
import importlib.util, subprocess, sys
if importlib.util.find_spec("ot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pot"])
print("POT available")


## Access — public format pair

CIMT Bonn cycling *dialogue* vs *survey* stand in for workshop vs survey when
the network is available. Assess without the analysis question: coverage by
format and unit sizes. If CIMT cannot be fetched, we fall back to a short
synthetic Access table so the notebook still runs.


In [ ]:
import os
from IPython.display import display
import pandas as pd
from multilogue.experiment import load_cimt_format_pair, format_tagged_units
from multilogue.assess_base import docs_per_group, unit_size_table


In [ ]:
os.environ.setdefault("MULTILOGUE_ACCESS_YES", "1")
cimt_units, cimt_meta = load_cimt_format_pair(max_units_per=60)
if cimt_units is None:
    print("CIMT unavailable:", cimt_meta.get("error", "unknown"))
    access_units = format_tagged_units(n_docs_per_group=3, units_per_doc=2)
    access_label = "synthetic format stand-in (Access fallback)"
else:
    access_units = cimt_units
    access_label = "CIMT Cycling-Dialogue-Bonn vs Survey-Bonn"
    display(cimt_meta)

print(access_label, "—", len(access_units), "units")
display(docs_per_group(access_units, "format"))
display(unit_size_table(access_units).describe())


German CIMT text does not match the English FakeClient phrase bank, so the
*Address* path below uses format-tagged synthetic English units with the same
group mechanics. With a real LLM client you can swap in the CIMT units from
Access; checkpoints still apply.


## Address — formats demo (FakeClient)

`run_formats_complexity_demo` wires the shared method end-to-end: multi-kind
extract, frozen clustering, multi-run lenses, hard and soft format contrasts,
$\sigma^2$ neighbour rules, document distance, and multiverse on the format-gap
$\ell$. No API key required.


In [ ]:
from pathlib import Path
from multilogue.experiment import run_formats_complexity_demo
from multilogue import viz
from multilogue.report import (
    cluster_catalogue, findings_markdown, top_phrases_by_group, write_findings,
)


In [ ]:
out = Path("outputs/notebook_formats_complexity")
summary = run_formats_complexity_demo(
    out, n_docs_per_group=4, units_per_doc=3, k=5, lens_runs=3, quiet=True,
)
display(summary)
tables = out / "tables"
for name in [
    "yield.csv",
    "group_compare_format_lenses.csv",
    "soft_group_compare_format.csv",
    "sigma2_rules.csv",
    "doc_distance_pairwise.csv",
    "variance_shares_format_gap.csv",
]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head(12))


In [ ]:
shares = pd.read_csv(tables / "variance_shares_format_gap.csv")
viz.variance_share_bars(shares, path=FIG / "09_format_gap_variance.png")
print("Soft:", summary.get("soft"))
print("Distance:", summary.get("distance"))


## Reading the run

Findings markdown summarises clusters, top concerns by format, and which
analytic decisions move $\ell$ — no extra LLM calls.


In [ ]:
from IPython.display import Markdown

ph = pd.read_csv(tables / "phrases_clustered.csv")
cmp = (
    pd.read_csv(tables / "group_compare_format_lenses.csv")
    if (tables / "group_compare_format_lenses.csv").is_file() else None
)
multi = (
    pd.read_csv(tables / "multiverse_format_gap.csv")
    if (tables / "multiverse_format_gap.csv").is_file() else None
)
shares = (
    pd.read_csv(tables / "variance_shares_format_gap.csv")
    if (tables / "variance_shares_format_gap.csv").is_file() else None
)
display(Markdown("### Cluster catalogue"))
display(cluster_catalogue(ph))
display(Markdown("### Top concerns by format"))
for g, tab in top_phrases_by_group(ph, kind="concern", n=8).items():
    display(Markdown(f"**{g}**"))
    display(tab)
if shares is not None:
    display(Markdown("### Multiverse variance shares"))
    display(shares)
md_text = findings_markdown(
    ph, compare=cmp, multiverse=multi, variance_shares=shares,
    title="Dialogue vs survey — findings",
)
write_findings(
    ph, tables / "findings.md", compare=cmp,
    multiverse=multi, variance_shares=shares,
    title="Dialogue vs survey — findings",
)
display(Markdown(md_text))


## Optional — live LLM (skipped without a key)

If `OPENAI_API_KEY` is set (for example via a local `.env`), the next cell runs
the same demo on CIMT units with a real client. Smoke tests and Colab-without-key
skip this cell's paid path automatically.


In [ ]:
import os
from pathlib import Path
from multilogue.config import load_dotenv

load_dotenv(Path.cwd() / ".env")
load_dotenv(Path.cwd().parent / ".env")
key = os.environ.get("OPENAI_API_KEY", "").strip()
if not key:
    print("No OPENAI_API_KEY — skipping live LLM path.")
elif cimt_units is None:
    print("CIMT units unavailable — skipping live LLM path.")
else:
    from multilogue.openai_client import OpenAIClient
    llm_out = Path("outputs/formats_cimt_llm")
    live = run_formats_complexity_demo(
        llm_out,
        units=[u for u in cimt_units if (u.get("meta") or {}).get("code") != "non-arg"],
        client=OpenAIClient(log_path=llm_out / "logs" / "openai.jsonl"),
        kinds=("concern", "benefit"),
        extract_model="gpt-4.1-mini",
        embed_model="text-embedding-3-small",
        label_model="gpt-4.1-mini",
        k=6, lens_runs=3, quiet=True, max_workers=4,
    )
    display(live)
    print("Checkpoints:", live.get("checkpoint_dir"))


# Outlook

Multilogue owns the shared stages above. A formats-style consumer package still
keeps: PDF/docx Access, study prompts and experience coding, researcher-edited
lens YAML, and the precise definition of headline $\ell$. See the formats
cookbook and the soft / EMD concepts page in the documentation.
